# Semantic Correspondence — Local (end-to-end)

This notebook runs **end-to-end on a local Linux machine** with an NVIDIA GTX 1080 Ti (11 GB VRAM):

1. Verify CUDA + GPU, apply hardware-specific optimizations
2. Check / download **SPair-71k** dataset
3. Download pretrained weights (DINOv2, DINOv3, SAM) if missing
4. Configure hyperparameters (batch sizes, epochs, precision) — **single cell**
5. Run the full pipeline (training + evaluation) with live dashboard
6. Display results: PCK tables, training curves, per-category and per-difficulty analysis, qualitative examples

**Artifacts** (logs, checkpoints, exports) are stored locally in `runs/` and `checkpoints/` (both gitignored).

### Prerequisites

```bash
# From the repository root, inside the .venv:
pip install -e ".[notebook]"          # Core + Jupyter + pandas
# Install CUDA-enabled PyTorch matching your driver:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
```

### 1. Runtime + CUDA sanity checks

In [ ]:
import os
import sys
from pathlib import Path

import torch

REPO_DIR = Path.cwd()
os.chdir(REPO_DIR)

print(f"Python     : {sys.executable}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
print(f"CUDA build : {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Install CUDA-enabled PyTorch:\n"
        "  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121"
    )

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_mem / (1024 ** 3)
cc = f"{props.major}.{props.minor}"

print(f"GPU        : {gpu_name}")
print(f"Compute cap: {cc}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"cuDNN      : {torch.backends.cudnn.version()}")

if "1080" not in gpu_name:
    print(f"\nWARNING: Expected GTX 1080 Ti but found '{gpu_name}'.")
    print("  Batch sizes in the config cell are tuned for 11 GB VRAM.")
    print("  Adjust FT_BATCH_SIZE_BY_BACKBONE / LORA_BATCH_SIZE_BY_BACKBONE if needed.")

if props.major < 7:
    print(f"\nNOTE: Compute capability {cc} (Pascal). bf16 is NOT supported.")
    print("  Using fp16 + GradScaler for VRAM savings (~2x smaller activations).")
    print("  fp16 does NOT speed up compute on Pascal (no Tensor Cores),")
    print("  but it allows larger batch sizes within the available VRAM.")

### 1b. CUDA optimizations

Enable cuDNN autotuner (profitable for fixed 784×784 inputs) and set matmul precision.

In [ ]:
import torch

# cuDNN autotuner: finds the fastest conv algorithm for fixed input sizes.
torch.backends.cudnn.benchmark = True

# TF32 is Ampere+ only; on Pascal these are harmless no-ops.
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
    torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except (RuntimeError, AttributeError):
    pass

allocated = torch.cuda.memory_allocated(0) / (1024 ** 2)
reserved = torch.cuda.memory_reserved(0) / (1024 ** 2)
print(f"VRAM baseline : {allocated:.0f} MB allocated, {reserved:.0f} MB reserved")
print("cuDNN benchmark: enabled")
print("CUDA optimizations applied.")

### 2. Dataset: SPair-71k

Check if `data/SPair-71k/` is present. If not, download and extract it (~1 GB).

In [ ]:
import os
import sys
import tarfile
import urllib.request
from pathlib import Path

SPAIR_URL = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
DATA_DIR = REPO_DIR / "data"
SPAIR_ROOT = DATA_DIR / "SPair-71k"
TAR_PATH = DATA_DIR / "SPair-71k.tar.gz"

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not SPAIR_ROOT.is_dir():
    if not TAR_PATH.is_file():
        print(f"Downloading: {SPAIR_URL}")
        urllib.request.urlretrieve(SPAIR_URL, TAR_PATH)
        print(f"Saved: {TAR_PATH}")
    else:
        print(f"Tarball already present: {TAR_PATH}")

    print(f"Extracting to: {DATA_DIR}")
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        if sys.version_info >= (3, 12):
            tar.extractall(path=DATA_DIR, filter="data")
        else:
            tar.extractall(path=DATA_DIR)
else:
    print(f"Dataset already present: {SPAIR_ROOT}")

print(f"SPAIR_ROOT = {SPAIR_ROOT} | exists = {SPAIR_ROOT.is_dir()}")

### 3. Pretrained weights (DINOv2, DINOv3, SAM)

Downloads pretrained ViT weights into `checkpoints/` if not already present.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "scripts/download_pretrained_weights.py"],
    cwd=str(REPO_DIR),
    check=True,
)

### 4. Configuration

**Edit this cell to change hyperparameters.** All batch sizes, epoch counts, and precision
settings are defined here and written to `config.yaml` for the pipeline.

#### GTX 1080 Ti guidance (11 GB VRAM, fp16)
| Backbone | FT batch | LoRA batch | Notes |
|----------|----------|------------|-------|
| DINOv2   | 16       | 16         | 784×784 input, patch 14 |
| DINOv3   | 16       | 16         | 784×784 input, patch 16 |
| SAM      | 2        | 2          | 1024×1024 internal, always tiny |

Reduce batch sizes if you get CUDA OOM. Increase if you have more VRAM.

In [ ]:
import yaml
from pathlib import Path

# ============================================================================
# USER-CONFIGURABLE PARAMETERS — edit these to taste
# ============================================================================

# --- Precision ---
# "fp16": saves ~50% VRAM via autocast + GradScaler (recommended for 1080 Ti)
# "fp32": full precision, needs smaller batches, no VRAM savings
# "auto": resolves to fp16 on Pascal (same as explicit fp16)
PRECISION = "fp16"

# --- Epochs & early stopping ---
FT_EPOCHS = 30           # Fine-tuning epochs (Colab uses 50; reduced for slower GPU)
FT_PATIENCE = 5          # Early stopping patience for fine-tuning
LORA_EPOCHS = 30         # LoRA epochs
LORA_PATIENCE = 5        # Early stopping patience for LoRA

# --- Batch sizes (per backbone, tuned for 1080 Ti 11 GB + fp16) ---
FT_BATCH_SIZE_BY_BACKBONE = {
    "dinov2_vitb14": 16,
    "dinov3_vitb16": 16,
    "sam_vit_b": 2,
}
LORA_BATCH_SIZE_BY_BACKBONE = {
    "dinov2_vitb14": 16,
    "dinov3_vitb16": 16,
    "sam_vit_b": 2,
}

# --- Learning rates ---
FT_LR = 5e-5
FT_WEIGHT_DECAY = 0.01
LORA_LR = 1e-3
LORA_ALPHA = 16.0
LORA_RANK = 8
LORA_LAST_BLOCKS = 2

# --- Fine-tuning depth sweep (Stage 2) ---
LAST_BLOCKS_LIST = [1, 2, 4]

# --- Pipeline behavior ---
START_FROM_SCRATCH = False   # True = clear pipeline_state.json (not checkpoints)
RUN_PYTEST = False           # True = run pytest after pipeline

# --- Image size (multiple of both 14 and 16 for ViT patch sizes) ---
IMAGE_HEIGHT = 784
IMAGE_WIDTH = 784

# --- Logging ---
LOG_BATCH_INTERVAL = 50      # Print batch progress every N steps
RESUME_SAVE_INTERVAL = 50    # Save resume checkpoint every N batches

# ============================================================================
# BUILD config.yaml — do not edit below this line
# ============================================================================

cfg_path = REPO_DIR / "config.yaml"

cfg = {
    "dataset": {
        "backend": "sd4match",
        "metrics_backend": "sd4match",
    },
    "paths": {
        "spair_root": str(REPO_DIR / "data" / "SPair-71k"),
        "checkpoint_dir": str(REPO_DIR / "checkpoints"),
    },
    "runtime": {
        "device": "cuda",
        "precision": PRECISION,
        "num_workers": -1,
        "preprocess": "FIXED_RESIZE",
        "image_height": IMAGE_HEIGHT,
        "image_width": IMAGE_WIDTH,
        "limit_pairs": 0,
        "eval_split": "test",
        "alphas": [0.05, 0.1, 0.2],
        "wsa_window": 5,
        "wsa_temperature": 1.0,
        "log_batch_interval": LOG_BATCH_INTERVAL,
        "resume_save_interval": RESUME_SAVE_INTERVAL,
    },
    "finetune": {
        "last_blocks": LAST_BLOCKS_LIST,
        "epochs": FT_EPOCHS,
        "patience": FT_PATIENCE,
        "batch_size": 16,
        "batch_size_by_backbone": FT_BATCH_SIZE_BY_BACKBONE,
        "lr": FT_LR,
        "weight_decay": FT_WEIGHT_DECAY,
        "dinov2_weights": str(REPO_DIR / "checkpoints" / "dinov2_vitb14_pretrain.pth"),
        "dinov3_weights": str(REPO_DIR / "checkpoints" / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"),
        "sam_checkpoint": str(REPO_DIR / "checkpoints" / "sam_vit_b_01ec64.pth"),
    },
    "lora": {
        "rank": LORA_RANK,
        "alpha": LORA_ALPHA,
        "lr": LORA_LR,
        "last_blocks": LORA_LAST_BLOCKS,
        "epochs": LORA_EPOCHS,
        "patience": LORA_PATIENCE,
        "batch_size": 16,
        "batch_size_by_backbone": LORA_BATCH_SIZE_BY_BACKBONE,
    },
    "workflow_toggles": {
        "run_verify_dataset": True,
        "train_finetune": [True, True, True],
        "train_lora": [True, True, True],
        "run_eval_baseline": [True, True, True],
        "run_eval_baseline_wsa": [True, True, True],
        "run_eval_finetuned_checkpoint": [True, True, True],
        "run_eval_lora_checkpoint": [True, True, True],
        "run_eval_finetuned_wsa": [True, True, True],
        "run_eval_lora_wsa": [True, True, True],
        "run_export_metrics_tables": True,
        "run_pytest": RUN_PYTEST,
        "pipeline_resume": not START_FROM_SCRATCH,
    },
}

with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

print(f"Written: {cfg_path}")
print(f"  precision          : {PRECISION}")
print(f"  finetune epochs    : {FT_EPOCHS} (patience {FT_PATIENCE})")
print(f"  lora epochs        : {LORA_EPOCHS} (patience {LORA_PATIENCE})")
print(f"  finetune batch map : {FT_BATCH_SIZE_BY_BACKBONE}")
print(f"  lora batch map     : {LORA_BATCH_SIZE_BY_BACKBONE}")
print(f"  checkpoint_dir     : {cfg['paths']['checkpoint_dir']}")
print(f"  start_from_scratch : {START_FROM_SCRATCH}")

### 5. Run pipeline

Launches `scripts/run_pipeline.py --config config.yaml` as a subprocess with a live dashboard
showing stage progress and the last 40 log lines, refreshed every 3 seconds.

**Expected runtime on 1080 Ti:** several hours for the full pipeline
(3 backbones × 3 block counts × fine-tune + LoRA + all evaluations).

In [ ]:
import json
import os
import subprocess
import sys
import threading
import time
from collections import deque
from pathlib import Path

from IPython.display import clear_output

os.chdir(REPO_DIR)

env = dict(os.environ)
if START_FROM_SCRATCH:
    env["SEMANTIC_CORRESPONDENCE_PIPELINE_RESET"] = "1"
else:
    env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_RESET", None)

env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_LOG_FILE_ONLY", None)
env["PYTHONUNBUFFERED"] = "1"

STAGE_EVENTS_PATH = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
TAIL_LINES = 40
REFRESH_SECONDS = 3.0

cmd = [sys.executable, "-u", "scripts/run_pipeline.py", "--config", "config.yaml"]

_output_lines: deque = deque(maxlen=TAIL_LINES)
_proc_done = threading.Event()
_return_code: list = []


def _stream_subprocess() -> None:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        _output_lines.append(line.rstrip())
    _return_code.append(proc.wait())
    _proc_done.set()


threading.Thread(target=_stream_subprocess, daemon=True).start()


def _read_stage_events() -> list:
    if not STAGE_EVENTS_PATH.is_file():
        return []
    events = []
    with open(STAGE_EVENTS_PATH, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            try:
                events.append(json.loads(raw))
            except json.JSONDecodeError:
                pass
    return events


def _render(events: list) -> None:
    done    = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed  = [e["stage_id"] for e in events if e.get("action") == "fail"]
    latest_start = next((e for e in reversed(events) if e.get("action") == "start"), None)
    current = latest_start["stage_id"] if latest_start else "(waiting...)"

    rc = _return_code[0] if _return_code else None
    if not _proc_done.is_set():
        status = "RUNNING"
    elif rc == 0:
        status = "COMPLETED"
    else:
        status = f"FAILED  (exit={rc})"

    sep = "=" * 64
    print(sep)
    print(f"  Pipeline: {status}")
    print(f"  Current stage : {current}")
    print(f"  Completed: {len(done):3d}  |  Skipped: {len(skipped):3d}  |  Failed: {len(failed):3d}")
    if done:
        tail = done[-6:]
        suffix = "..." if len(done) > 6 else ""
        print(f"  Last done: {', '.join(tail)}{suffix}")
    if failed:
        print(f"  !! FAILED: {', '.join(failed)}")
    print(sep)
    print(f"--- last {TAIL_LINES} log lines ---")
    for line in _output_lines:
        print(line)


while not _proc_done.is_set():
    clear_output(wait=True)
    _render(_read_stage_events())
    _proc_done.wait(timeout=REFRESH_SECONDS)

clear_output(wait=True)
_render(_read_stage_events())

rc = _return_code[0] if _return_code else -1
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
print("\nPipeline completed successfully.")

### 5b. Post-run stage summary (optional)

Re-display the final stage summary from `runs/logs/stage_events.jsonl`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

STAGE_EVENTS_PATH = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"

if not STAGE_EVENTS_PATH.is_file():
    print(f"No stage events file found at: {STAGE_EVENTS_PATH}")
else:
    events = []
    with open(STAGE_EVENTS_PATH, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if raw:
                try:
                    events.append(json.loads(raw))
                except json.JSONDecodeError:
                    pass

    done    = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed  = [e["stage_id"] for e in events if e.get("action") == "fail"]

    print(f"Total stage events : {len(events)}")
    print(f"Completed          : {len(done)}")
    print(f"Skipped            : {len(skipped)}")
    print(f"Failed             : {len(failed)}")
    if failed:
        print(f"!! Failed stages: {failed}")

    display(pd.DataFrame(events).tail(20))

### 5c. Training curves (loss vs epoch)

Plot train and validation loss over epochs for every fine-tune and LoRA run.
History files (`*_history.jsonl`) are saved alongside checkpoints.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

CKPT_DIR = REPO_DIR / "checkpoints"

history_files = sorted(CKPT_DIR.glob("*_history.jsonl"))
if not history_files:
    print("No training history files found in", CKPT_DIR)
else:
    print(f"Found {len(history_files)} history file(s):")
    histories = {}
    for hf in history_files:
        name = hf.stem.replace("_history", "")
        print(f"  {hf.name}")
        records = []
        with open(hf, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        if records:
            histories[name] = records

    if histories:
        n = len(histories)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
        for col, (name, records) in enumerate(sorted(histories.items())):
            ax = axes[0, col]
            epochs = [r["epoch"] for r in records]
            ax.plot(epochs, [r["train_loss"] for r in records], marker=".", label="train_loss")
            ax.plot(epochs, [r["val_loss"] for r in records], marker=".", label="val_loss")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Loss")
            ax.set_title(name, fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        fig.suptitle("Training curves", fontsize=13)
        fig.tight_layout()
        plt.show()
    else:
        print("History files found but all empty.")

---

## Results Analysis

### 6. Load pipeline exports

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

os.chdir(REPO_DIR)
EXPORTS = REPO_DIR / "runs" / "pipeline_exports"


def _load_json(name):
    p = EXPORTS / name
    if p.is_file():
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    print(f"Not found: {p}")
    return None


pck_results = _load_json("pck_results.json")
per_category = _load_json("pck_results_per_category.json")
by_difficulty = _load_json("pck_results_by_difficulty_flag.json")

print(f"Loaded {len(pck_results or [])} evaluation runs.")

### 7. Aggregate PCK comparison table

In [ ]:
if pck_results:
    rows = []
    for r in pck_results:
        row = {"name": r["name"]}
        row.update(r.get("metrics", {}))
        rows.append(row)
    df_pck = pd.DataFrame(rows)
    display(
        df_pck.style.format(
            {c: "{:.4f}" for c in df_pck.columns if c.startswith("pck@")}
        ).set_caption("PCK Results")
    )
else:
    print("No results to display.")

### 8. Fine-tuning depth comparison (Stage 2)

In [ ]:
import re

import matplotlib.pyplot as plt

if pck_results:
    ft_rows = []
    for r in pck_results:
        m = re.match(r"(.+)_ft_lb(\d+)$", r["name"])
        if m:
            backbone = m.group(1)
            lb = int(m.group(2))
            row = {"backbone": backbone, "last_blocks": lb}
            row.update(r.get("metrics", {}))
            ft_rows.append(row)

    if ft_rows:
        df_ft = pd.DataFrame(ft_rows).sort_values(["backbone", "last_blocks"])
        display(df_ft)

        fig, axes = plt.subplots(
            1,
            len(df_ft["backbone"].unique()),
            figsize=(5 * len(df_ft["backbone"].unique()), 4),
            squeeze=False,
        )
        for col_idx, (bb, grp) in enumerate(df_ft.groupby("backbone")):
            ax = axes[0, col_idx]
            metric_cols = [c for c in grp.columns if c.startswith("pck@")]
            for mc in metric_cols:
                ax.plot(grp["last_blocks"], grp[mc], marker="o", label=mc)
            ax.set_xlabel("Unfrozen blocks")
            ax.set_ylabel("PCK")
            ax.set_title(bb)
            ax.legend(fontsize=8)
            ax.set_xticks(sorted(grp["last_blocks"].unique()))
        fig.suptitle("PCK vs. number of fine-tuned blocks", fontsize=13)
        fig.tight_layout()
        plt.show()
    else:
        print("No fine-tuned checkpoint results found (names must match *_ft_lb<N>).")

### 9. Per-category PCK breakdown

In [ ]:
import numpy as np

if per_category:
    rows = []
    for entry in per_category:
        for cat, alphas in entry.get("categories", {}).items():
            row = {"run": entry["name"], "category": cat}
            row.update(alphas)
            rows.append(row)
    if rows:
        df_cat = pd.DataFrame(rows)
        pck_col = [c for c in df_cat.columns if c.startswith("pck@")]
        if pck_col:
            pivot = df_cat.pivot_table(
                index="category", columns="run", values=pck_col[0], aggfunc="first"
            )
            fig, ax = plt.subplots(
                figsize=(
                    max(12, len(pivot.columns) * 1.5),
                    max(6, len(pivot) * 0.4),
                )
            )
            im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index, fontsize=8)
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    v = pivot.values[i, j]
                    if not np.isnan(v):
                        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
            fig.colorbar(im, ax=ax, label=pck_col[0])
            ax.set_title(f"Per-category {pck_col[0]}")
            fig.tight_layout()
            plt.show()
    else:
        print("No per-category data available.")
else:
    print("Per-category export not found.")

### 10. Per-difficulty analysis

In [ ]:
if by_difficulty:
    rows = []
    for entry in by_difficulty:
        run_name = entry["name"]
        for flag, buckets in entry.get("data", {}).items():
            for bucket, info in buckets.items():
                summary = info.get("summary", {}).get("image", {})
                for metric_key, val_dict in summary.items():
                    row = {
                        "run": run_name,
                        "flag": flag,
                        "value": int(bucket),
                        "metric": metric_key.replace("custom_", ""),
                        "pck": val_dict.get("all", float("nan")),
                    }
                    rows.append(row)
    if rows:
        df_diff = pd.DataFrame(rows)
        display(
            df_diff.pivot_table(
                index=["run", "metric"], columns=["flag", "value"], values="pck"
            ).round(4)
        )
    else:
        print("No difficulty breakdown data.")
else:
    print("Difficulty export not found.")

### 11. Qualitative correspondence examples

In [ ]:
import matplotlib.pyplot as plt
import torch
from data.dataset import PreprocessMode, SPair71kPairDataset
from evaluation.visualize import visualize_correspondences
from models.common import (
    BackboneName,
    DenseExtractorConfig,
    DenseFeatureExtractor,
    predict_correspondences_cosine_argmax,
)

# Free VRAM from any prior pipeline work before loading the model.
torch.cuda.empty_cache()

spair_root = str(REPO_DIR / "data" / "SPair-71k")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ds = SPair71kPairDataset(
    spair_root=spair_root,
    split="test",
    preprocess=PreprocessMode.FIXED_RESIZE,
    output_size_hw=(784, 784),
    normalize=True,
)

cfg_ext = DenseExtractorConfig(
    name=BackboneName.DINOV2_VIT_B14,
    dinov2_weights_path=str(REPO_DIR / "checkpoints" / "dinov2_vitb14_pretrain.pth"),
)
extractor = DenseFeatureExtractor(cfg_ext).to(device).eval()

NUM_EXAMPLES = 4
indices = list(
    range(0, min(len(ds), NUM_EXAMPLES * 50), max(1, len(ds) // NUM_EXAMPLES))
)[:NUM_EXAMPLES]

for idx in indices:
    sample = ds[idx]
    src = sample["src_img"].unsqueeze(0).to(device)
    tgt = sample["tgt_img"].unsqueeze(0).to(device)
    src_kps = sample["src_kps"].to(device)
    tgt_kps = sample["tgt_kps"].to(device)
    pck_thr = float(sample["pck_threshold_bbox"])

    with torch.no_grad():
        feat_src, meta_src = extractor(src)
        feat_tgt, meta_tgt = extractor(tgt)
        out = predict_correspondences_cosine_argmax(
            feat_src,
            feat_tgt,
            src_kps,
            img_hw=(784, 784),
            img_hw_src=(784, 784),
            img_hw_tgt=(784, 784),
        )
        pred_tgt = out["pred_tgt_xy"]

    fig = visualize_correspondences(
        sample["src_img"],
        sample["tgt_img"],
        src_kps,
        pred_tgt,
        tgt_kps,
        pck_threshold=pck_thr,
        alpha=0.1,
        title=f"Pair {idx} (category: {sample.get('category', '?')})",
    )
    plt.show()
    plt.close(fig)